In [1]:
!nvidia-smi

Mon May 25 11:51:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.12              Driver Version: 550.90.12      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               Off |   00000000:00:07.0 Off |                    0 |
| 44%   59C    P8             31W /  300W |       2MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Important constants
HF_REPO_NAME = "loknezmonzter/Ministral-3-3B-Instruct-2512-BF16-FT-PMC-Distilled"
# DATASET_PATH = "/mnt/huggingface/data/distilled/pmc_patients/pmc-patients-distilled-medgemma-22B"
BASE_DIR = "checkpoints/"
FINE_TUNED_MODEL = "Ministral-3-3B-Instruct-2512-BF16-FT-PMC-Distilled"
MAX_SEQ_LENGTH = 4096
NUM_TRAIN_EPOCHS = 3 
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 5e-5
SEED = 3407

In [4]:
import torch
from transformers import (
    Mistral3ForConditionalGeneration,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "mistralai/Ministral-3-3B-Instruct-2512-BF16"

# Same BnB config that WORKS for you
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# Same loading path that WORKS for you
model = Mistral3ForConditionalGeneration.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Prepare for k-bit training (required before adding LoRA)
model = prepare_model_for_kbit_training(model)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/45.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/198k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/147k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

In [3]:
# Print all linear layer names - very important
# Need to know exact names of linear layers for fine tuning
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        print(name)

model.vision_tower.transformer.layers.0.feed_forward.gate_proj
model.vision_tower.transformer.layers.0.feed_forward.up_proj
model.vision_tower.transformer.layers.0.feed_forward.down_proj
model.vision_tower.transformer.layers.0.attention.k_proj
model.vision_tower.transformer.layers.0.attention.v_proj
model.vision_tower.transformer.layers.0.attention.q_proj
model.vision_tower.transformer.layers.0.attention.o_proj
model.vision_tower.transformer.layers.1.feed_forward.gate_proj
model.vision_tower.transformer.layers.1.feed_forward.up_proj
model.vision_tower.transformer.layers.1.feed_forward.down_proj
model.vision_tower.transformer.layers.1.attention.k_proj
model.vision_tower.transformer.layers.1.attention.v_proj
model.vision_tower.transformer.layers.1.attention.q_proj
model.vision_tower.transformer.layers.1.attention.o_proj
model.vision_tower.transformer.layers.2.feed_forward.gate_proj
model.vision_tower.transformer.layers.2.feed_forward.up_proj
model.vision_tower.transformer.layers.2.feed_f

In [5]:
# Target only the linear layers of language model
language_modules = [
    # Attention — "self_attn" only exists in language_model, not vision_tower
    "self_attn.q_proj",
    "self_attn.k_proj",
    "self_attn.v_proj",
    "self_attn.o_proj",
    # MLP — "mlp" only exists in language_model; vision_tower uses "feed_forward"
    "mlp.gate_proj",
    "mlp.up_proj",
    "mlp.down_proj",
]

In [6]:
# Configure LoRA for fine tuning
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=language_modules,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

# Verify: count how many modules got LoRA adapters
lora_params = sum(p.numel() for n, p in model.named_parameters() if "lora" in n)
total_params = sum(p.numel() for p in model.parameters())
print(f"LoRA params: {lora_params:,} / {total_params:,} ({100*lora_params/total_params:.2f}%)")

# Verify no vision tower modules were targeted
for n, _ in model.named_modules():
    if "vision_tower" in n and "lora" in n:
        print(f"WARNING: Vision tower module got LoRA: {n}")

LoRA params: 49,414,144 / 2,175,693,824 (2.27%)


In [23]:
from peft.tuners.lora import LoraLayer

lora_modules = []
for name, module in model.named_modules():
    if isinstance(module, LoraLayer):
        lora_modules.append(name)

print(f"Total LoRA-targeted modules: {len(lora_modules)}")
print()

# Group by layer type
vision_matches = [m for m in lora_modules if "vision_tower" in m]
lang_matches   = [m for m in lora_modules if "language_model" in m]
other_matches  = [m for m in lora_modules if "vision_tower" not in m and "language_model" not in m]

print(f"Vision tower modules targeted: {len(vision_matches)}  ← should be 0")
print(f"Language model modules targeted: {len(lang_matches)}  ← should be 182")
print(f"Other modules targeted: {len(other_matches)}  ← should be 0")

if vision_matches:
    print(f"\n⚠️  WARNING: Vision tower leaked! {vision_matches[:3]}...")

Total LoRA-targeted modules: 182

Vision tower modules targeted: 0  ← should be 0
Language model modules targeted: 182  ← should be 182
Other modules targeted: 0  ← should be 0


In [24]:
# Verify padding token exists for tokenizer
if tokenizer.pad_token is None:
    print(tokenizer.pad_token)
else:
    print("NO PADDING TOKEN FOUND! APPLYING PADDING...")
    tokenizer.pad_token = tokenizer.eos_token

# Required: causal LM training needs right-side padding
tokenizer.padding_side = "right"

NO PADDING TOKEN FOUND! APPLYING PADDING...


In [9]:
tokenizer.pad_token

'</s>'

In [7]:
import json

# Exact JSON schema - convert to JSON string
JSON_SCHEMA = {
    "summary": "A concise, 1-2 sentence abstractive summary of the clinical scenario.",
    "clinical_reasoning": "A step-by-step logical breakdown of the diagnoses, treatments, or clinical decisions made in the text. Explain WHY certain relationships exist. Keep short and brief but to the point.",
    "relationships": [
        {
            "subject": "Source entity (e.g., Patient, Drug, Symptom)",
            "predicate": "Use STANDARD POSITIVE relationships (e.g., HAS_HISTORY, SHOWS_SYMPTOM, DIAGNOSED_WITH, PRESCRIBED). Do not use negated verbs like 'DENIES' or 'LACKS'.",
            "object": "Target entity",
            "polarity": "positive OR negative (Use 'negative' if the patient denies the history or lacks the symptom)",
            "certainty": "confirmed, suspected, OR hedged",
            "evidence": "The exact verbatim text snippet that proves this relationship."
        }
    ],
    "keywords": ["List", "of", "important", "clinical", "NER", "terms"]
}
SCHEMA_STRING = json.dumps(JSON_SCHEMA, indent=4)

# Finalized system prompt
SYSTEM_PROMPT = (
    "You are an expert clinical informatician. "
    f"Extract data strictly into this JSON schema:\n\n{SCHEMA_STRING}\n\n"
    "CRITICAL RULES:\n"
    "1. Use ONLY double quotes for all JSON keys and string values.\n"
    "2. Response MUST start with '{' and end with '}'.\n"
    "3. Output raw JSON only — no markdown, no code blocks.\n"
    "4. Provide values for ALL keys in the schema.\n"
    "5. Apostrophes in clinical terms (e.g., patient's) are allowed inside double-quoted strings.\n"
    "6. Extract at max 10 most clinically significant relationships only. "
    "Prioritize: diagnosis > treatment > symptoms > history.\n"
)

In [25]:
def format_example(example):
    """
    Format columns into standard Hugging Face conversation dictionaries.
    The output dict MUST expose a single structural column names "messages"
    """
    # Ground truth from medgemma extractions
    target_json = json.dumps(example['data'], indent=2)

     # Build prompt string with chat template (includes assistant header)
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"CONTEXT:\n{example['text']}"}
    ]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True  # Adds the assistant role header (e.g., <|assistant|>)
    )

    # Completion is just the JSON output + EOS
    completion_text = target_json + tokenizer.eos_token

    return {
        "prompt": prompt_text,      
        "completion": completion_text,
    }

In [ ]:
def format_example_with_ids(example):
    target_json = json.dumps(example['data'], indent=2)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"CONTEXT:\n\n{example['text']}\n"},
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False,
    )
    completion_text = target_json + tokenizer.eos_token

    # Tokenize separately to know the exact boundary
    prompt_ids = tokenizer(prompt_text)["input_ids"]
    completion_ids = tokenizer(completion_text, add_special_tokens=False)["input_ids"]

    # Build input_ids and labels with explicit masking
    input_ids = prompt_ids + completion_ids
    labels = [-100] * len(prompt_ids) + completion_ids

    return {
        "input_ids": input_ids,
        "labels": labels,
        "prompt_len": len(prompt_ids),  # for debugging / collation
    }

In [27]:
from dotenv import load_dotenv
from datasets import load_dataset

load_dotenv()

# Load the dataset
dataset = load_dataset(
    "loknezmonzter/pmc-patients-distilled-medgemma-22B",
    split="train",
    revision="distilled-medgemma-15k"
)

# Randomize the train dataset 
# Split entire train set into smaller train and test
train = dataset.shuffle(seed=3407) 
split = train.train_test_split(test_size=0.25, seed=6174)

# Use 1650 random records for training and 200 random records for evaluation
train_split = split["train"].select(range(2500))
eval_split = split["test"].select(range(600, 850))

trainx = train_split.map(format_example, desc="Formatting train")
evalx = eval_split.map(format_example, desc="Formatting eval")

print("\nsuccessfully prepared train and test sets")
print(f"rows in train: {len(trainx)}")
print(f"rows in eval: {len(evalx)}\n")

Formatting train:   0%|          | 0/2500 [00:00<?, ? examples/s]

Formatting eval:   0%|          | 0/250 [00:00<?, ? examples/s]


successfully prepared train and test sets
rows in train: 2500
rows in eval: 250



In [30]:
print(trainx[0]['completion'])

{
  "summary": "A 62-year-old male with a history of hypertension, hyperlipidemia, atherosclerosis, type II diabetes, and extensive smoking presented with acute vomiting and severe abdominal pain. Imaging and subsequent laparoscopic cholecystectomy revealed a distended gallbladder with calculi, a 3.0 cm polypoid mass, focal wall thickening, and metastatic lymph node involvement, ultimately diagnosed as LCNEC.",
  "clinical_reasoning": "The patient presented with symptoms suggestive of biliary colic or cholecystitis (vomiting, RUQ/epigastric pain). Initial ultrasound showed hepatomegaly and cholelithiasis but no cholecystitis. A noncontrast CT scan revealed concerning findings (stones, possible polyp, wall thickening). Laparoscopic cholecystectomy was performed due to these findings. Pathology confirmed a large polypoid mass with metastatic lymph node involvement, consistent with LCNEC based on immunohistochemistry. The patient recovered postoperatively and was discharged without chemot

In [14]:
import numpy as np

full_texts = [p + c for p, c in zip(train_set["prompt"], train_set["completion"])]
token_counts = [len(tokenizer.tokenize(text)) for text in full_texts]
p95 = np.percentile(token_counts, 95)

print(f"95% of full sequences in TRAIN SET are shorter than {p95:.0f} tokens.")

95% of full sequences in TRAIN SET are shorter than 2818 tokens.


In [15]:
import numpy as np

full_texts = [p + c for p, c in zip(eval_set["prompt"], eval_set["completion"])]
token_counts = [len(tokenizer.tokenize(text)) for text in full_texts]
p95 = np.percentile(token_counts, 95)

print(f"95% of full sequences in EVAL SET are shorter than {p95:.0f} tokens.")

95% of full sequences in EVAL SET are shorter than 2959 tokens.


In [ ]:
prompt_tokens = [len(tokenizer.tokenize(p)) for p in train_set["text"]]
compl_tokens = [len(tokenizer.tokenize(c)) for c in train_set["completion"]]

print(f"Prompt: 95th percentile = {np.percentile(prompt_tokens, 95):.0f} tokens")
print(f"Completion: 95th percentile = {np.percentile(compl_tokens, 95):.0f} tokens")

Prompt: 95th percentile = 1625 tokens
Completion: 95th percentile = 1225 tokens


In [28]:
import uuid
from datetime import datetime

# Config pieces
run_type   = "train"                                   # or "eval", "sft", "dpo", etc.
_model = "ministral3"
_model = _model.replace("-", "_")               # sanitize for filesystem/URL safety

# Timestamp: 20250525_132745
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Optional short hash (first 6 chars of a UUID): e.g., a3f9b2
short_hash = uuid.uuid4().hex[:6]

# Pick one format:
run_name = f"{run_type}_{_model}_{timestamp}"
# run_name = f"{run_type}_{model_name}_{timestamp}_{short_hash}"   # if you want extra uniqueness

print("Run name:", run_name)

Run name: train_ministral3_20260525_112410


In [26]:
def calculate_warmup_steps():
    effective_batch_size = BATCH_SIZE * GRAD_ACCUM_STEPS
    steps_per_epoch = len(train_set) // effective_batch_size
    total_steps = steps_per_epoch * NUM_TRAIN_EPOCHS
    warmup_steps = round(total_steps * 0.03)
    return warmup_steps

warmpup_steps = calculate_warmup_steps()
warmpup_steps

7

In [50]:
# Check that masking is correct
sample = train[0]
full_text = sample["prompt"] + sample["completion"]
tokens = tokenizer(full_text, return_tensors="pt")
prompt_len = len(tokenizer(sample["prompt"], return_tensors="pt")["input_ids"][0])

print(f"Full sequence: {tokens['input_ids'].shape[1]} tokens")
print(f"Prompt: {prompt_len} tokens")
print(f"Completion: {tokens['input_ids'].shape[1] - prompt_len} tokens")
# The completion token count should match your earlier 1246-ish number

Full sequence: 1943 tokens
Prompt: 912 tokens
Completion: 1031 tokens


In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorWithPadding

# Training configuration for fine tuning
training_args = SFTConfig(
    output_dir=f"checkpoints/{FINE_TUNED_MODEL}", 

    # --- Set step count for train --- #
    num_train_epochs=3,
    max_steps=-1, # ensures enough optimizer steps to give a clear picture of dry run

    # --- Batch configuration --- #
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    eval_accumulation_steps=4, 
    # gradient_checkpointing=True,
    # gradient_checkpointing_kwargs={"use_reentrant": False},

    # --- Optimizations --- #
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_grad_norm=1.0,
    optim="adamw_8bit",

    # --- Precision control --- #
    bf16=True,

    # --- Masking --- #
    completion_only_loss=False,

    # --- Checkpointing --- #
    save_steps=38,
    eval_steps=38,
    save_strategy="steps",
    eval_strategy="steps",
    save_total_limit=6,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # --- Logging and reporting --- #
    logging_steps=5,

    seed=SEED,
)

collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=trainx,
    eval_dataset=evalx,
    processing_class=tokenizer,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/2500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2500 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

In [32]:
try:
    trainer.train(resume_from_checkpoint=False)
except Exception as e:
    print(f"\n\n[ERROR]\n\n{e}\n\n")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1, 'pad_token_id': 11}.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: loknezmonzter (loknezmonzter-dev) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
38,1.137981,1.065432,1.124249,1244712.000000,0.756897
76,1.033854,1.036923,1.076269,2489188.000000,0.761521
114,0.979900,1.028686,1.055415,3705128.000000,0.762681
152,0.995979,1.017942,1.049266,4949669.000000,0.763292
190,0.931769,1.013049,1.034136,6165430.000000,0.763494
228,0.998238,1.011907,1.036363,7409951.000000,0.763679
237,1.043594,1.011924,1.036451,7676190.000000,0.763613


In [2]:
!nvidia-smi

Mon May 25 06:27:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.12              Driver Version: 550.90.12      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               Off |   00000000:00:07.0 Off |                    0 |
| 30%   31C    P8             25W /  300W |       2MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [35]:
wandb.finish()

ConnectionResetError: Connection lost